# 03 — Import des documents économiques locaux

Transforme les PDF et classeurs Excel de la bibliothèque ECO RCI en faits structurés compatibles avec le même nettoyage que les pages web. La validation rejette ou répare les cellules manifestement décalées (valeur mal groupée, unité contaminée, libellé coupé ou période aberrante) avant qu'elles puissent devenir des chunks ou des embeddings.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


In [1]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


Projet : <PROJECT_ROOT>
Python : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA\.venv\Scripts\python.exe
Données actives : <PROJECT_ROOT>\data


In [2]:
import pandas as pd
from dataclasses import asdict
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
source_dir = config.local_documents.source_full_path
files = sorted(path for path in source_dir.rglob('*') if path.is_file())
pdfs = [path for path in files if path.suffix.casefold() == '.pdf']
spreadsheets = [path for path in files if path.suffix.casefold() in {'.xlsx', '.xlsm'}]
supported_documents = pdfs + spreadsheets
display(pd.Series({
    'dossier_source': str(source_dir),
    'fichiers': len(files),
    'pdf': len(pdfs),
    'tableurs': len(spreadsheets),
    'autres_fichiers': len(files) - len(supported_documents),
    'taille_Mo': sum(path.stat().st_size for path in files) / 1024**2,
}).to_frame('valeur'))
display(pd.DataFrame({
    'source': [path.relative_to(source_dir).parts[0] for path in supported_documents],
    'catégorie': [path.relative_to(source_dir).parent.as_posix() for path in supported_documents],
    'format': [path.suffix.casefold() for path in supported_documents],
}).groupby(['source', 'catégorie', 'format']).size().rename('documents').to_frame())


,valeur
dossier_source,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...
fichiers,84
pdf,80
tableurs,4
autres_fichiers,0
taille_Mo,195.254552


documents
source              catégorie                                          format           
ANSTAT              ANSTAT/Comptes_Nationaux                           .pdf            9
                    ANSTAT/Comptes_Nationaux_Excel                     .xlsx           4
                    ANSTAT/INDICE DE PRIX DE PRODUCTION DE L’INDUST... .pdf           25
                    ANSTAT/INDICE DU CHIFFRE D'AFFAIRES DANS LE COM... .pdf            4
                    ANSTAT/INDICE HARMONISE DES PRIX A LA CONSOMMAT... .pdf            9
                    ANSTAT/Recensements_RGPH                           .pdf            3
                    ANSTAT/RÉPERTOIRE NATIONAL DES ENTREPRISES DE C... .pdf            2
                    ANSTAT/Statistiques_Generales_CIV                  .pdf            2
                    ANSTAT/TABLEAU DE BORD SOCIO-ECONOMIQUE (TBSE) ... .pdf            9
AUTRES_INSTITUTIONS AUTRES_INSTITUTIONS/Accords_FMI_Memorandums        .pdf            5
                    AUTRES_INSTITUTIONS/Cadrages_Macroeconomiques      .pdf            5
                    AUTRES_INSTITUTIONS/Finances_Publiques             .pdf            7

In [3]:
from local_documents import LocalPDFImporter

RUN_LOCAL_PDF_IMPORT = True
if RUN_LOCAL_PDF_IMPORT:
    report = LocalPDFImporter(config).import_all(show_progress=True)
    display(pd.Series(asdict(report)).to_frame('valeur'))
    if report.failures:
        display(pd.DataFrame(report.failures))
    assert report.discovered_pdfs > 0
    success_ratio = (report.imported_pdfs + report.reused_pdfs) / report.discovered_pdfs
    assert success_ratio >= 0.90, 'Moins de 90 % des PDF ont pu être importés.'
    if report.failed_pdfs:
        print(f'AVERTISSEMENT : {report.failed_pdfs} PDF restent indisponibles et nécessitent un OCR ou un nouveau téléchargement.')
    assert report.discovered_spreadsheets > 0, 'Aucun tableur local détecté.'
    spreadsheet_success = report.imported_spreadsheets + report.reused_spreadsheets
    assert spreadsheet_success == report.discovered_spreadsheets, 'Certains tableurs n’ont pas été importés.'


Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
PDF non importé : ANSTAT\Recensements_RGPH\RESUTATS GLOBAUX DEFINITIFS RGPH 2021.pdf — ValueError: aucun texte exploitable ; OCR probablement nécessaire
PDF non importé : ANSTAT\RÉPERTOIRE NATIONAL DES ENTREPRISES DE CÔTE D'IVOIRE 2024\CENTRALE DE BILANS 2023.pdf — ValueError: aucun texte exploitable ; OCR probablement nécessaire
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 44 0 (offset 0)
Ignoring wrong pointing object 46 0 (offset 

,valeur
discovered_pdfs,80
imported_pdfs,74
reused_pdfs,0
duplicate_pdfs,0
failed_pdfs,6
discovered_spreadsheets,4
imported_spreadsheets,0
reused_spreadsheets,4
duplicate_spreadsheets,0
failed_spreadsheets,0


,path,error
0,ANSTAT/Recensements_RGPH/RESUTATS GLOBAUX DEFI...,ValueError: aucun texte exploitable ; OCR prob...
1,ANSTAT/RÉPERTOIRE NATIONAL DES ENTREPRISES DE ...,ValueError: aucun texte exploitable ; OCR prob...
2,AUTRES_INSTITUTIONS/Cadrages_Macroeconomiques/...,ValueError: aucun texte exploitable ; OCR prob...
3,AUTRES_INSTITUTIONS/Cadrages_Macroeconomiques/...,ValueError: aucun texte exploitable ; OCR prob...
4,AUTRES_INSTITUTIONS/Cadrages_Macroeconomiques/...,ValueError: aucun texte exploitable ; OCR prob...
5,AUTRES_INSTITUTIONS/Finances_Publiques/Indicat...,ValueError: aucun texte exploitable ; OCR prob...


AVERTISSEMENT : 6 PDF restent indisponibles et nécessitent un OCR ou un nouveau téléchargement.


## Résultat transmis à la suite

Deux barres de progression affichent le PDF ou le tableur en cours, les pages ou feuilles déjà lues, le moteur d'extraction et les compteurs d'import, de réutilisation, de doublons et d'erreurs.

Les originaux restent dans `data/sources/eco_rci`. Les records régénérables `local_pdf_*.json` et `local_sheet_*.json` sont ajoutés à `data/raw`. Leur URL pointe vers l'API locale afin que les citations ouvrent le document original.

Les feuilles Excel sont segmentées à chaque nouvel en-tête temporel, puis transformées dans le même contrat de faits atomiques que les tableaux PDF. Le rapport complet est écrit dans `data/import_reports/eco_rci.json`.


## Extraction structurée des tableaux statistiques

En plus du texte des pages, l'importeur reconstruit les cellules des tableaux sous la forme
`indicateur + période + valeur + unité + territoire + type de mesure + page`. Les coordonnées visuelles sont utilisées :
une cellule vide ne décale donc plus les valeurs vers la mauvaise année. Les tableaux paysages tournés de 90° sont remis dans leur repère logique avant extraction.

In [4]:
import json
from table_facts import is_plausible_table_fact

raw_pdf_records = [
    json.loads(path.read_text(encoding='utf-8'))
    for pattern in ('local_pdf_*.json', 'local_sheet_*.json')
    for path in sorted(config.local_documents.raw_full_path.glob(pattern))
]
table_rows = []
for record in raw_pdf_records:
    for fact in record.get('table_facts', []):
        table_rows.append({
            'document': record.get('title', ''),
            'url': record.get('url', ''),
            **fact,
        })
table_facts_df = pd.DataFrame(table_rows)
display(pd.Series({
    'documents_locaux_importés': len(raw_pdf_records),
    'faits_tabulaires': len(table_facts_df),
    'pages_avec_tableaux': table_facts_df[['url', 'page']].drop_duplicates().shape[0] if not table_facts_df.empty else 0,
    'indicateurs_distincts': table_facts_df['indicator'].nunique() if not table_facts_df.empty else 0,
    'faits_depuis_tableurs': int(table_facts_df['source_method'].eq('spreadsheet_matrix').sum()) if not table_facts_df.empty else 0,
}).to_frame('valeur'))
assert not table_facts_df.empty, 'Aucun fait tabulaire n’a été extrait des documents locaux.'
assert table_facts_df['source_method'].eq('spreadsheet_matrix').any(), 'Aucun fait issu des feuilles Excel n’a été extrait.'
invalid_facts = [fact for record in raw_pdf_records for fact in record.get('table_facts', []) if not is_plausible_table_fact(fact)]
assert not invalid_facts, f'{len(invalid_facts)} faits tabulaires mal formés ont franchi l’import.'

critical_facts = table_facts_df[
    table_facts_df['indicator'].str.casefold().isin({'population homme', 'rapport de masculinité'})
    & table_facts_df['period'].astype(str).eq('2025')
]
display(critical_facts[['document', 'page', 'domain', 'indicator', 'period', 'value', 'unit', 'territory', 'measure_type']])
expected = {
    ('population homme', '2025', '17 103 895'),
    ('rapport de masculinité', '2025', '109,3'),
}
observed = set(zip(
    critical_facts['indicator'].str.casefold(),
    critical_facts['period'].astype(str),
    critical_facts['value'].astype(str),
))
assert expected <= observed, 'Les deux faits de contrôle de la page 10 ne sont pas correctement extraits.'

quarterly = table_facts_df[
    table_facts_df['document'].str.contains('trimestr', case=False, na=False)
    & table_facts_df['page'].isin([14, 17])
    & table_facts_df['period'].astype(str).str.contains('2025', na=False)
].copy()
display(quarterly[['document', 'page', 'indicator', 'period', 'value', 'unit', 'territory', 'measure_type']].head(30))
pib_t4_volume = quarterly[
    quarterly['page'].eq(14)
    & quarterly['indicator'].str.casefold().eq('pib')
    & quarterly['period'].astype(str).str.contains(r'T4.*2025', regex=True)
]
assert '12 083,3' in set(pib_t4_volume['value']), 'Le PIB T4 2025 de la page paysage 14 est absent.'
assert pib_t4_volume['measure_type'].str.contains('CVS').all()
assert pib_t4_volume['measure_type'].str.contains('volume').all()
assert set(quarterly['territory']) == {"Cote d'Ivoire"}
print('Contrôle tabulaire validé : valeurs, périodes, pages et facettes sont explicitement reliées.')

,valeur
documents_locaux_importés,78
faits_tabulaires,56090
pages_avec_tableaux,499
indicateurs_distincts,4149
faits_depuis_tableurs,11760


,document,page,domain,indicator,period,value,unit,territory,measure_type
4833,Chiffres Cles CIV 2025,10,DÉMOGRAPHIE,Population homme,2025,17 103 895,Nombre,Cote d'Ivoire,
4855,Chiffres Cles CIV 2025,10,DÉMOGRAPHIE,Rapport de masculinité,2025,"109,3",Nombre,Cote d'Ivoire,


,document,page,indicator,period,value,unit,territory,measure_type
9561,Comptes nationaux trimestriels (1er trimestrie...,14,PRIMAIRE,T1 2025,"2 001,6",milliards de FCFA,Cote d'Ivoire,CVS | volume | prix chaines de 2015
9562,Comptes nationaux trimestriels (1er trimestrie...,14,PRIMAIRE,T2 2025,"2 098,7",milliards de FCFA,Cote d'Ivoire,CVS | volume | prix chaines de 2015
9563,Comptes nationaux trimestriels (1er trimestrie...,14,PRIMAIRE,T3 2025,"2 086,5",milliards de FCFA,Cote d'Ivoire,CVS | volume | prix chaines de 2015
9564,Comptes nationaux trimestriels (1er trimestrie...,14,PRIMAIRE,T4 2025,"2 089,6",milliards de FCFA,Cote d'Ivoire,CVS | volume | prix chaines de 2015
9566,Comptes nationaux trimestriels (1er trimestrie...,14,AGRICULTURE VIVRIERE,T1 2025,"529,1",milliards de FCFA,Cote d'Ivoire,CVS | volume | prix chaines de 2015
9567,Comptes nationaux trimestriels (1er trimestrie...,14,AGRICULTURE VIVRIERE,T2 2025,"521,1",milliards de FCFA,Cote d'Ivoire,CVS | volume | prix chaines de 2015
9568,Comptes nationaux trimestriels (1er trimestrie...,14,AGRICULTURE VIVRIERE,T3 2025,"530,8",milliards de FCFA,Cote d'Ivoire,CVS | volume | prix chaines de 2015
9569,Comptes nationaux trimestriels (1er trimestrie...,14,AGRICULTURE VIVRIERE,T4 2025,"567,4",milliards de FCFA,Cote d'Ivoire,CVS | volume | prix chaines de 2015
9571,Comptes nationaux trimestriels (1er trimestrie...,14,AGRICULTURE D'EXPORTATION,T1 2025,"1 113,0",milliards de FCFA,Cote d'Ivoire,CVS | volume | prix chaines de 2015
9572,Comptes nationaux trimestriels (1er trimestrie...,14,AGRICULTURE D'EXPORTATION,T2 2025,"1 223,7",milliards de FCFA,Cote d'Ivoire,CVS | volume | prix chaines de 2015


Contrôle tabulaire validé : valeurs, périodes, pages et facettes sont explicitement reliées.
